# Self-Attention in a Transformer

**Full single-head attention module at the bottom.**

---

Notebook implements a basic version of a single-head causal self-attention block:

input tensor $X$ \
↓ \
projection matrices $W_Q,W_K,W_V$ \
↓ \
compute $Q,K,V$ \
↓ \
compute (raw) attention scores $Q \cdot K^T$ \
↓ \
scale by dividing by $\sqrt {d_{k}}$ \
↓ \
implement and apply causal mask \
↓ \
normalize with softmax \
↓ \
weighted sums with $V$

In [ ]:
import torch
import torch.nn as nn

Input shape:
#### $[1,4,8]$

---

Before everything, we'll make a fake/sample input $X$ with an reasonable, inspectable size.

In [ ]:
batch_size = 1 # B
sequence_length = 4 # T
d_model = 8 # D

# Random input data
x = torch.randn(
    batch_size,
    sequence_length, # token (position) count
    d_model # number of features in each token
)

print("Input tensor shape:", x.shape)

## Query, Key, Value Matrices

### Projections

We'll start off with initializing the projection matrices $W_Q,W_K,W_V$.

Commonly in practice, $Q,K,V$ are one matrix to save resources, being size $[d_{model}, 3d_{model}]$, which are then split after.

---

`Linear()` is PyTorch's implementation of a **learned** linear transformation $y = xW^T + b$.

https://docs.pytorch.org/docs/2.14/generated/torch.nn.Linear.html

In [ ]:
q_proj = nn.Linear(d_model, d_model, bias=False) # keep dim, no added bias
k_proj = nn.Linear(d_model, d_model, bias=False)
v_proj = nn.Linear(d_model, d_model, bias=False)

Computing $Q,K,V$, which are the learned matrices that represent every token in a sequence. The feature values in each matrix are all different, in practice that means they are all **independently** learned.

$Q=XW_Q$

$K=XW_K$

$V=XW_V$

In [ ]:
q = q_proj(x)
k = k_proj(x)
v = v_proj(x)

print("Q shape:", q.shape)
print("K shape:", k.shape)
print("V shape:", v.shape)

# Verify features differ
print("\nSame Q and K:", torch.equal(q, k))
print("Same K and V:", torch.equal(k, v))


Ouput shapes should be the same as that matters in computing the (dot-product) attention scores.

PyTorch displays weights as $[d_\text{input}, d_\text{output}]$.

In [ ]:
print("Q weight shape:", q_proj.weight.shape)
print("K weight shape:", k_proj.weight.shape)
print("V weight shape:", v_proj.weight.shape)

## Calculating Raw Attention Scores

### Why Compute Dot Product
In order to find what token vector representations influence another and by how much, we compute the dot product of every possible pair of tokens. This results in the raw attention scores.

The dot product of 2 vectors (token representations in this case) tells you how much they point in the same direction:
- A **large positive** value denotes the vectors align well with each other, in self-attention it signifies that a vector should be paid more attention (have a higher influence).

- A **low** or **negative** value denotes the vectors don't have a strong relationship, signifying lower influence.

---

### Tensor Shapes

Computing the dot product (raw scores) requires **inner dimensions** to match i.e. the number of **columns** in the first matrix to equal the number of **rows** in the second matrix.

The result takes the **outer dimensions**, for example: \
$\begin{bmatrix} a & b & c \end{bmatrix}_{1 \times 3 \ \text{matrix}} \begin{bmatrix} d \\ e \\ f \end{bmatrix}_{3 \times 1 \ \text{matrix}} = \begin{bmatrix} ad + be + cf \end{bmatrix}_{1 \times 1 \ \text{matrix}}$

To get the scores for $Q$ and $K$, we'll **transpose** $K$ into $K^T$: \
$Q: [B, T, d_{model}]$ \
$K^T: [B, d_{model}, T]$

So the tensor shapes are: \
$ [B, T, d_{model}] \times [B, d_{model}, T] \rightarrow [B, T, T]$, the batch $B$ dimension stays untouched.

https://docs.pytorch.org/docs/2.14/generated/torch.transpose.html

In [ ]:
# Dot product of Q and K^T, batch dimension is not touched
scores = q @ k.transpose(-2, -1) # transpose second last (-2) and last (-1) dimensions

print("Q shape:", q.shape)
print("K^T shape:", k.transpose(-2, -1).shape)

# Raw attention scores
print("\nScore tensor shape:", scores.shape)

# Each row is a q vector, each column is a k vector
print("\nRaw Scores:\n", scores)

We can verify the scores by manually computing the dot product of $q_0$ and $k_1$ and comparing it to the same corresponding position.

The score here literally represents: how strong token $0$'s **query** matches to token $1$'s **key**.

In [ ]:
manual_score = torch.dot(q[0, 0], k[0, 1])

print("Manual score:", manual_score)
print("Matrix score:", scores[0, 0, 1])

### Scaling Attention Scores Pre-Softmax

The raw scores are then divided $\frac{QK^T}{\sqrt{d_k}}$, because:
- Raw attention scores grow larger as $d_k$ increases.
- Dividing by $\sqrt{d_k}$ keeps the scores at a **stable** scale so when softmax performs $e^\text{score}$, the result doesn't explode.
- This prevents softmax distributions from becoming too **extreme**, where one token gets nearly all the attention.

In [ ]:
import math

# Size of last dimension
d_k = q.size(-1) # we could also use k.size(-1), as they have the same shape

# Scale raw scores with sqrt(d_k)
scaled_scores = scores / math.sqrt(d_k)

print("Scaled scores shape:", scaled_scores.shape)
print("\nScaled scores:\n", scaled_scores)

print("\nRaw scores for reference:\n", scores)

## Causal Mask

### Why Implement Causal Masking?

In autoregressive LMs, we apply **causal masking** during training because we don't want the current token position to attend to tokens in the future.

Suppose we were at token position 2, we only want the attention mechanism to use token 2 and 1 i.e. **itself** and **previous** tokens, in order to make predictions.

During **inference**, a model won't have "future" data help it make predictions, which is why **causal masking** is used give the model the same environmental restriction when **training**.

---

In generally, a masked attention-score matrix will look like this after causal masking is applied:

$\begin{array}{c|ccccc} & \vec{k}_0 & \vec{k}_1 & \vec{k}_2 & \cdots & \vec{k}_n \\\hline\vec{q}_0 & * & -\infin & -\infin & \cdots & -\infin \\\vec{q}_1 & * & * & -\infin & \cdots & -\infin \\\vec{q}_2 & * & * & * & \cdots & -\infin \\\vdots & \vdots & \vdots & \vdots & \ddots & \vdots \\\vec{q}_n & * & * & * & \cdots & *\end{array}$

https://docs.pytorch.org/docs/2.14/generated/torch.tril.html \
https://docs.pytorch.org/docs/2.14/generated/torch.ones.html

In [ ]:
def make_causal_mask(sequence_length):
    return torch.tril( # upper triangular filled with 0s
        torch.ones(sequence_length, sequence_length) # [T, T] filled with 1s
    )

The function return a causally masked matrix, with its **lower triangle filled with 1s**. \
Intuitively, it means:

$q_1$ can see $k_0$ \
$q_2$ can see keys $k_{0,1}$ \
$q_2$ can see keys $k_{0,1,2}$ \
$q_3$ can see keys $k_{0,1,2,3}$

---

So for the **values** in the mask matrix is are just a triangular shaped **binary mask**:

$1$: this token is allowed to attend here \
$0$: this token cannot attend here

In [ ]:
mask = make_causal_mask(sequence_length)

print(mask)

### Applying the Mask

The mask is now applied to the scaled attention score matrix. The causally masked matrix created was essentially a bonafide **boolean mask**.

https://docs.pytorch.org/docs/2.14/generated/torch.Tensor.masked_fill_.html

---

`-inf` is a special floating-point value **representing** negative infinity. It isn't an infinitely large number stored in memory. \
Instead, a reserved bit pattern tells the computer to **treat** the value according to **mathematical rules** for $-\infin$

In [ ]:
# Causally mask future positions, lower triangle stays untouched
masked_scores = scaled_scores.masked_fill(
    mask == 0, # where mask is 0
    float("-inf") # replace with -infinity
)

print(masked_scores)

## Softmax

Softmax normalizes every score to be between 0-1, with every **row** summing to 1. This now gives the attention distribution for each **query** token across every **key** token.

$\text{Softmax}(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$, where $i$ is the current score, j iterates over all scores (and sums them).

For any values that are `-inf`, they become $e^{-\infin} = 0$

---

A causally-masked matrix will generally look like this after softmax:

$\begin{array}{c|ccccc} & \vec{k}_0 & \vec{k}_1 & \vec{k}_2 & \cdots & \vec{k}_n \\\hline\vec{q}_0 & 1 & 0 & 0 & \cdots & 0 \\\vec{q}_1 & * & * & 0 & \cdots & 0 \\\vec{q}_2 & * & * & * & \cdots & 0 \\\vdots & \vdots & \vdots & \vdots & \ddots & \vdots \\\vec{q}_n & * & * & * & \cdots & *\end{array}$ \
Notice the only nonzero score in the **first** row (token position) becomes $1$.

https://docs.pytorch.org/docs/2.14/generated/torch.nn.Softmax.html

In [ ]:
# Soft each row, across each row's values (columns)
attention_weights = torch.softmax(
    masked_scores,
    dim=-1 # apply across last dimension
)

print("Attention weights:\n", attention_weights)

print("\nCheck each rows sum to 1:\n", attention_weights.sum(dim=-1))

## Output Weighted Sums With $V$

The attention weights matrix $A$ is now multiplied by the value matrix $V$.

Conceptually if token 2 has attention ($a_1$) weights like: $[0.2, 0.5, 0.3, 0.0]$

Then the output would be: \
$O_1 = 0.2v_0 + 0.5v_1 + 0.3v_2 + 0.0v_3$

---

The tensor goes back to the model's original input $X$'s shape: 

$A:[B, T, T]$ \
$V:[B, T, d_k]$

↓

$AV:[B,T,d_k]$ \
Because $[\mathbf{T},T] \times [T,\mathbf{d_k}] \rightarrow [\mathbf{T,d_k}]$ (result takes outer dims)

In [ ]:
attention_output = attention_weights @ v

print("Attention output shape:", attention_output.shape)

Conceptually if token 2 has attention ($a_1$) weights like: $[0.2, 0.5, 0.3, 0.0]$

Then the output would be: \
$O = 0.2v_0 + 0.5v_1 + 0.3v_2 + 0.0v_3$

Notice how each there's $4$ token vectors that have dimensions $8$, which are wrapped in $1$ batch.

Output shape:
#### $[1,4,8]$

In [ ]:
print("Attention output:\n", attention_output)

print("\nInput shape:", x.shape)
print("Output shape:", attention_output.shape)

# Output same as input
assert attention_output.shape == x.shape

## Full Single-Head Attention Class

In [ ]:
class SingleHeadCausalAttention(nn.Module):
    """
    Single-head causal self-attention block
    """

    def __init__(self, d_model):
        super().__init__()

        self.d_model = d_model

        self.q_proj = nn.Linear(d_model, d_model, bias=False) # no bias for simplicity purposes
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        # Project input into query, key, value representations
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # Dot product to getraw attention scores
        scores = q @ k.transpose(-2, -1)

        # Scale raw scores
        d_k = q.size(-1)
        scores = scores / math.sqrt(d_k)

        # Causal mask based on input sequence length
        sequence_length = x.size(1) # second dim

        mask = torch.tril( # upper triangle set false
            torch.ones(
                sequence_length,
                sequence_length,
                device=x.device, # make sure tensors stay on same device i.e. CPU/GPU
                dtype=torch.bool # boolean-type mask
            )
        )

        scores = scores.masked_fill(
            ~mask, # inverse the mask as true values get masked
            float("-inf")
        )

        # Apply softmax to get attention weights
        attention_weights = torch.softmax(
            scores,
            dim=-1
        )

        # Weighted sum of value vectors
        output = attention_weights @ v

        return output

Below we can test the class using the same input.

--- 
Calling an nn.Module automatically runs its forward() method. \
PyTorch states:

"Although the recipe for forward pass needs to be defined within this function, one should call the Module instance afterwards instead of this since the former takes care of running the registered hooks while the latter silently ignores them."

https://docs.pytorch.org/docs/2.14/generated/torch.nn.Module.html

In [ ]:
attention = SingleHeadCausalAttention(
    d_model=d_model
)

output = attention(x)

assert output.shape == x.shape

print("Input shape:", x.shape)
print("Output shape:", output.shape)